In [ ]:
SEED = 42
OUTPUT_FILE = "old_code_results.csv"

In [ ]:
import random
import numpy as np
import torch

def set_seed(seed_value):
    """Sets the seed for reproducibility."""
    random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed_value)
        # The two lines below are for ensuring deterministic behavior on GPU
        # but can sometimes have a performance cost.
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

# Set the seed for your experiment
set_seed(SEED)

In [1]:
from pathlib import Path
import pickle
import sys
import os
sys.path.append('../')

DATA_DIR = Path('../data/100000_data_points')
DATA_PATH = DATA_DIR / "synthetic_rl_datasets.pkl"
try:
    with open(DATA_PATH, 'rb') as f:
        all_data = pickle.load(f)
except FileNotFoundError:
    print(f"File not found: {DATA_PATH}")
except Exception as e:
    print(f"Error loading dataset: {e}")


In [2]:
import random
import numpy as np
import sys
import torch
from tqdm import tqdm
import os
import matplotlib.pyplot as plt

sys.path.append('../')
# Add the root directory to the Python path to find your modules

from models.reward_cnn import RewardCNNCentralized
from tests.cnn.centralized.generate_synthetic_states import (
    generate_synthetic_state_at_most_1_apples,
)
from config import MODEL_DIR, GRAPHS_DIR
import torch
from tests.cnn.centralized.generate_synthetic_states import (
    generate_synthetic_state_at_most_1_apples,
)
from train_scripts.train_centralized_cnn import MODEL_SAVE_PATH
import matplotlib.pyplot as plt
from config import GRAPHS_DIR, OUT_DIR
from tqdm import tqdm




--- PyTorch is configured to use: cuda ---


In [3]:
import json
import sys
sys.path.append('../')

from matplotlib.ticker import NullLocator, ScalarFormatter
from decentralized_helpers import generate_reward_vector, pick_agent_uniformly, get_agent_positions
from models.reward_cnn import RewardCNNDecentralized



In [ ]:
import json

from numpy import dtype, float64, ndarray
from models.reward_cnn import RewardCNNDecentralized
from helpers.controllers import ViewController
from decentralized_helpers import generate_reward_vector, pick_agent_uniformly, get_agent_positions


key = '6x6_2_agents'
hidden = 128
accuracies = []
mae_overall_list = []
mae_neg1_list = []
mae_zero_list = []
mape_positive_list = [] # Using MAPE for positive rewards
mape_positive_list = [] # Using MAPE for positive rewards


p_pick_apple = 0.5



print(f"--- probability: {p_pick_apple} ---")
data = all_data[key]

NUM_DATA_POINTS = 100000
num_picks_apple = int(NUM_DATA_POINTS * p_pick_apple)
num_no_picks_apple = NUM_DATA_POINTS - num_picks_apple

data_with_apple = data['picks_apple'][:num_picks_apple]
data_without_apple = data['doesnt_pick_apple'][:num_no_picks_apple]

state_agentpos_reward = data_with_apple + data_without_apple
random.shuffle(state_agentpos_reward)
# print(len(state_agentpos_reward))
# print(state_agentpos_reward[0])

NUM_TRAIN_EPISODES = 1000
percent_train = 0.8
total_train = int(0.8 * NUM_DATA_POINTS)
state_agentpos_reward_TRAIN = state_agentpos_reward[:total_train]
state_agentpos_reward_TEST = state_agentpos_reward[total_train:]
if key == '6x6_2_agents':
    WIDTH = 6
    HEIGHT = 6
    NUM_AGENTS = 2
elif key == '9x9_4_agents':
    WIDTH = 9
    HEIGHT = 9
    NUM_AGENTS = 4
elif key == '12x12_7_agents':
    WIDTH = 12
    HEIGHT = 12
    NUM_AGENTS = 7
else:
    raise ValueError(f"Unknown key: {key}")
BATCH_SIZE = 32
lr = 0.01

NUM_ACTING_AGENTS = NUM_AGENTS

# we use a list of networks instead of one
networks = []

for _ in range(NUM_AGENTS):
    networks.append(RewardCNNDecentralized(HEIGHT, WIDTH, lr, mlp_hidden_features=hidden))
print(type(networks[0]))

for i in tqdm(range(NUM_TRAIN_EPISODES), desc="Training"):
    for b in range(BATCH_SIZE):
        row_index = i * BATCH_SIZE + b

        # Pick the constant that represents how many acting agents we have. In the first setup, it's always agent with id 0.
        picker_id = pick_agent_uniformly(NUM_ACTING_AGENTS)

        assert(picker_id < NUM_ACTING_AGENTS)

        # We know that apple and agent are in the same cell if the apple is picked
        apple_pos = state_agentpos_reward_TRAIN[row_index]["agent_pos"]


        agent_positions = get_agent_positions(state_agentpos_reward_TRAIN[row_index]["state"]["agents"],
                                                state_agentpos_reward_TRAIN[row_index]["agent_pos"], picker_id)

        picked =  state_agentpos_reward_TRAIN[row_index]["reward"] != 0

        assert np.array_equal(agent_positions[picker_id], apple_pos)
        reward_vector: ndarray[tuple[int], dtype[float64]] | ndarray[tuple[int, ...], dtype[float64]] = generate_reward_vector(picker_id, agent_positions, apple_pos, picked)
        if picked:
            assert(reward_vector[picker_id] == -1)
            sum_ = 0
            for r in reward_vector:
                if r != -1:
                    sum_ += r

            # check with tolerance
            assert np.isclose(sum_, 2.0, atol=1e-8)
        else:
            assert(reward_vector[picker_id] == 0)

        state = state_agentpos_reward_TRAIN[row_index]["state"]

        for a in range(NUM_AGENTS):
            reward = reward_vector[a]
            reward_float = np.float32(reward)
            position = agent_positions[a]
            networks[a].add_experience_from_raw(state, reward, agent_pos=position)

    for model in networks:
        model.train_batch()

model = networks[0]
# print(f"Final loss after training: {model.loss_history[-1]}")

# plot model loss
# plt.figure(figsize=(8, 6))
# plt.plot(model.loss_history)
# plt.xlabel("Training Iterations")
# plt.ylabel("Loss")
# plt.title("Model Loss Over Time")
# plt.savefig(GRAPHS_DIR / f"cnn_decentralized_loss_allActing_{key}_{hidden}h.png")

################## TESTING ##################
num_test_episodes = len(state_agentpos_reward_TEST)
num_correct = 0
num_correct_when_reward_negative = 0
num_correct_when_reward_0 = 0
num_correct_when_reward_positive = 0
num_negative_predictions = 0
num_zero_predictions = 0
num_positive_predictions = 0
tol = 0.01 
errors = []
errors_reward_negative = []
errors_reward_0 = []
errors_reward_positive = []
relative_errors_positive = [] 
for i in tqdm(range(num_test_episodes), leave=False, desc="Testing"):
    picker_id = pick_agent_uniformly(NUM_ACTING_AGENTS)

    assert(picker_id < NUM_ACTING_AGENTS)

    # We know that apple and agent are in the same cell if the apple is picked
    apple_pos = state_agentpos_reward_TEST[i]["agent_pos"]

    agent_positions = get_agent_positions(state_agentpos_reward_TEST[i]["state"]["agents"],
                                            state_agentpos_reward_TEST[i]["agent_pos"], picker_id)

    picked =  state_agentpos_reward_TEST[i]["reward"] != 0 # BUG should be TEST not TRAIN

    assert np.array_equal(agent_positions[picker_id], apple_pos)
    reward_vector = generate_reward_vector(picker_id, agent_positions, apple_pos, picked)
    if picked:
        assert(reward_vector[picker_id] == -1)
        sum_ = 0
        for r in reward_vector:
            if r != -1:
                sum_ += r
        # check with tolerance
        assert np.isclose(sum_, 2.0, atol=1e-8)
    else:
        assert(reward_vector[picker_id] == 0)

    state = state_agentpos_reward_TEST[i]["state"]
    a= 0 # only evaluate agent 0 for consistency
    reward = reward_vector[a]
    position = agent_positions[a]
    prediction = networks[a].get_model_reward_prediction_from_raw(state, agent_pos=position).item()
    error = abs(prediction - reward)
    errors.append(error)
    if reward == -1:
        errors_reward_negative.append(error)
        num_negative_predictions += 1
        if error < tol:
                num_correct_when_reward_negative += 1
    elif reward == 0:
        errors_reward_0.append(error)
        num_zero_predictions += 1
        if error < tol:
            num_correct_when_reward_0 += 1
    elif reward > 0:
        errors_reward_positive.append(error)
        num_positive_predictions += 1
        relative_errors_positive.append(error / reward)
        if error < tol:
            num_correct_when_reward_positive += 1
    if error < tol:
        num_correct += 1

mape_positive = (np.mean(relative_errors_positive) * 100) if relative_errors_positive else 0
mape_positive_list.append(mape_positive)
    
# --- Append metrics for ERROR plot ---
mae_overall_list.append(np.mean(errors) if errors else 0)
mae_neg1_list.append(np.mean(errors_reward_negative) if errors_reward_negative else 0)
mae_zero_list.append(np.mean(errors_reward_0) if errors_reward_0 else 0)


--- probability: 0.5 ---
<class 'models.reward_cnn.RewardCNNDecentralized'>


Training: 100%|██████████| 1000/1000 [00:07<00:00, 137.03it/s]


In [ ]:
mape_positive = (np.mean(relative_errors_positive) * 100) if relative_errors_positive else 0
mae_overall = np.mean(errors) if errors else 0
mae_neg1 = np.mean(errors_reward_negative) if errors_reward_negative else 0
mae_zero = np.mean(errors_reward_0) if errors_reward_0 else 0
mae_positive = np.mean(errors_reward_positive) if errors_reward_positive else 0

# ===================================================================
# NEW PART: Instead of appending to a list in memory,
# append the results for this run to the persistent output file.
# ===================================================================
import os

# Create the file and write the header if it doesn't exist
if not os.path.exists(OUTPUT_FILE):
    with open(OUTPUT_FILE, 'w') as f:
        f.write("Seed,MAE_overall,MAE_reward_minus_1,MAE_reward_0,MAPE_positive\n")

# Append the results of the current run as a new row
with open(OUTPUT_FILE, 'a') as f:
    f.write(f"{SEED},{mae_overall},{mae_neg1},{mae_zero},{mae_positive},{mape_positive}\n")

print(f"Results for seed {SEED} saved to {OUTPUT_FILE}")

200010000


KeyboardInterrupt: 